In [1]:
import numpy as np
import parismc
import dynesty
import matplotlib.pyplot as plt

In [ ]:
import os
import sys
# Change to the desired directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/sampling_test')

# Add it to Python path
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/sampling_test')

In [3]:
def loglike(params):
    params = np.asarray(params)

    n_samples = params.shape[0]
    log_likes = np.zeros(n_samples)

    for i in range(n_samples):
        logm1, logm2, a, p0, e0, dist, cosqS, phiS, Phi_phi0, Phi_r0 = params[i]
        m1 = 10**logm1
        m2 = 10**logm2
        qS = np.arccos(cosqS)
        phiK = phiS + np.pi/3

        htemp = waveform_gen(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK,
                            Phi_phi0, Phi_theta0, Phi_r0, T=T, dt=dt)
        
        res = data - htemp
        res_f = gwf.freq_wave(res)
        inner_res = gwf.inner(res_f, res_f)
        log_likes[i] = -0.5 * inner_res

    return log_likes


In [ ]:
def prior_transform(u):
    logm1lim = [5.9998955889e+00, 6.0001044111e+00]
    logm2lim = [1.4770586003e+00, 1.4771839092e+00]
    alim = [6.9991943608e-01, 7.0008056392e-01]
    p0lim = [7.4995568044e+00, 7.5004431956e+00]
    e0lim = [3.9997901137e-01, 4.0002098863e-01]
    distlim = [1.9727323083e+00, 2.0272676917e+00]
    cosqSlim = [8.6479203952e-01, 8.9037308426e-01]
    phiSlim = [9.7750458220e-01, 1.0224954178e+00]
    Phiphilim = [3.8232596200e-01, 4.1767403800e-01]
    Phirlim = [4.9151222865e-01, 5.0848777135e-01]

    transformed = np.zeros_like(u)

    # Uniform in log for masses

    # m1
    transformed[:, 0] = (logm1lim[1] - logm1lim[0]) * u[:, 0] + logm1lim[0]

    # m2
    transformed[:, 1] = (logm2lim[1] - logm2lim[0]) * u[:, 1] + logm2lim[0]

    # Linear in others 

    # a
    transformed[:, 2] = (alim[1] - alim[0]) * u[:, 2] + alim[0]

    # p0
    transformed[:, 3] = (p0lim[1] - p0lim[0]) * u[:, 3] + p0lim[0] 

    # e0
    transformed[:, 4] = (e0lim[1] - e0lim[0]) * u[:, 4] + e0lim[0]

    # dist 
    transformed[:, 5] = (distlim[1] - distlim[0]) * u[:, 5] + distlim[0]

    # Uniform in cosqS 

    # qS
    transformed[:, 6] = (cosqSlim[1] - cosqSlim[0]) * u[:, 6] + cosqSlim[0]

    # phiS
    transformed[:, 7] = (phiSlim[1] - phiSlim[0]) * u[:, 7] + phiSlim[0]

    # Phi_phi0
    transformed[:, 8] = (Phiphilim[1] - Phiphilim[0]) * u[:, 8] + Phiphilim[0]

    # Phi_r0
    transformed[:, 9] = (Phirlim[1] - Phirlim[0]) * u[:, 9] + Phirlim[0]

    
    return transformed

In [5]:
paris_sampler = parismc.Sampler.load_state('./paris_3s_results/sampler_state.pkl')

In [ ]:
def compare_marginal(sampler, savepath, dynesty_file, true_values=None):
    """
    Create marginal distribution plots comparing PARIS and Dynesty 
results
    """
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        import pickle
    except ImportError:
        print("Required packages not available. Skipping visualization.")
        return

    print("\nCreating marginal distribution comparison plots...")

    # Get PARIS samples and weights
    paris_samples, paris_weights = sampler.get_samples_with_weights(flatten=True)
    ndim = paris_samples.shape[1]

    # Load Dynesty results
    try:
        with open(dynesty_file, 'rb') as f:
            dres = pickle.load(f)
        dynesty_samples, dynesty_weights = dres.samples, dres.importance_weights()
        # mean, cov = dyfunc.mean_and_cov(dynesty_samples, dynesty_weights)
        print(f"Loaded Dynesty results from {dynesty_file}")
    except Exception as e:
        print(f"Error loading Dynesty results: {e}")
        return

    # Parameter ranges from prior_transform function
    param_ranges = [
        [5.9998955889e+00, 6.0001044111e+00],
        [1.4770586003e+00, 1.4771839092e+00],
        [6.9991943608e-01, 7.0008056392e-01],
        [7.4995568044e+00, 7.5004431956e+00],
        [3.9997901137e-01, 4.0002098863e-01],
        [1.9727323083e+00, 2.0272676917e+00],
        [8.6479203952e-01, 8.9037308426e-01],
        [9.7750458220e-01, 1.0224954178e+00],
        [3.8232596200e-01, 4.1767403800e-01],
        [4.9151222865e-01, 5.0848777135e-01]
    ]

    # Parameter names for titles
    param_names = ['logm1', 'logm2', 'a', 'p0', 'e0', 'dist', 'cosqS', 'phiS','Phi_phi0', 'Phi_r0']

    # Visualization parameters
    bin_num = 50
    decay = 3  # For exponential smoothing

    def exponential_smoothing(hist, decay=1.0):
        """Apply exponential smoothing to histogram."""
        smoothed = np.zeros_like(hist)
        for i in range(len(hist)):
            weights_exp = np.exp(-decay * np.abs(np.arange(len(hist)) -i))
            weights_exp /= np.sum(weights_exp)  # normalize
            smoothed[i] = np.sum(hist * weights_exp)
        return smoothed

    # Set up the plot
    sns.set(style="white", context="talk")

    # Create subplot grid
    n_rows = (ndim + 1) // 2
    fig, axes = plt.subplots(n_rows, 2, figsize=(24, 3*n_rows))

    # Flatten axes for easier indexing
    if ndim == 1:
        axes = [axes]
    elif ndim <= 2:
        axes = axes.flatten()
    else:
        axes = axes.flatten()

    for i in range(ndim):
        ax = axes[i]

        # PARIS samples
        paris_param_samples = paris_samples[:, i]
        paris_hist, bin_edges = np.histogram(paris_param_samples,bins=bin_num, weights=paris_weights, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        paris_hist = exponential_smoothing(paris_hist, decay=decay)
        ax.plot(bin_centers, paris_hist, color='green', linewidth=2,label='PARIS')

        # Dynesty samples
        if i < dynesty_samples.shape[1]:
            dynesty_param_samples = dynesty_samples[:, i]
            dynesty_hist, bin_edges = np.histogram(dynesty_param_samples,bins=bin_num, weights=dynesty_weights, density=True)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            dynesty_hist = exponential_smoothing(dynesty_hist,decay=decay)
            ax.plot(bin_centers, dynesty_hist, color='blue', linewidth=2,label='Dynesty')

        # Add true value red line if provided
        if true_values is not None and i < len(true_values):
            ax.axvline(true_values[i], color='red', linestyle='--',linewidth=2,
                    label='True Value' if i == 0 else "")

        # Set x-axis limits to the prior ranges
        if i < len(param_ranges):
            ax.set_xlim(param_ranges[i][0], param_ranges[i][1])

        # Formatting with variable names
        param_name = param_names[i] if i < len(param_names) else f'Param {i+1}'
        ax.set_title(f'{param_name} Marginal Distribution', fontsize=14)
        ax.set_ylabel('Density', fontsize=12)
        ax.grid(True, alpha=0.3)

        # Add legend to first subplot
        if i == 0:
            ax.legend(fontsize=10, frameon=True)

    # Remove empty subplots if ndim is odd
    if ndim % 2 == 1 and ndim > 1:
        fig.delaxes(axes[-1])

    # Adjust layout
    plt.tight_layout()

    # Save the plot
    plot_filename = os.path.join(savepath,'compare_marginal.png')
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Marginal distribution comparison plot saved to: {plot_filename}")

    # Show plot if in interactive environment
    try:
        plt.show()
    except:
        pass

In [4]:
m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 7.5
e0 = 0.4 
xI0 = 1.0 #NOTE: fixed
dist = 2 # Gpc
# Polar and azimuthal angles .. detector frame
# S = Solar system barycenter
# K = spin angular momentum of the MBH
qS = 0.5 
phiS = 1 
qK = 1 #NOTE: fixed
phiK = phiS + np.pi/3
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # NOTE: fixed
Phi_r0 = 0.5
param_true_linear = np.array([m1, m2, a, p0, e0, dist, qS, phiS, Phi_phi0, Phi_r0])
param_true = np.array([np.log10(m1), np.log10(m2), a, p0, e0, dist, np.cos(qS), phiS, Phi_phi0, Phi_r0])

In [6]:
compare_marginal(paris_sampler, '.', true_values=param_true, dynesty_file='ns_likelihoodtest.pkl')

In [20]:
def compare_marginal_linear(sampler, savepath, dynesty_file, true_values=None):
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        import pickle
    except ImportError:
        print("Required packages not available. Skipping visualization.")
        return

    print("\nCreating marginal distribution comparison plots...")

    # Get PARIS samples and weights
    paris_samples, paris_weights = sampler.get_samples_with_weights(flatten=True)
    ndim = paris_samples.shape[1]

    # Load Dynesty results
    try:
        with open(dynesty_file, 'rb') as f:
            dres = pickle.load(f)
        dynesty_samples, dynesty_weights = dres.samples, dres.importance_weights()
        # mean, cov = dyfunc.mean_and_cov(dynesty_samples, dynesty_weights)
        print(f"Loaded Dynesty results from {dynesty_file}")
    except Exception as e:
        print(f"Error loading Dynesty results: {e}")
        return
    
    # Transform samples to physical parameters
    # Convert log10(m1) -> m1, log10(m2) -> m2, cos(qS) -> qS
    paris_samples_transformed = paris_samples.copy()
    dynesty_samples_transformed = dynesty_samples.copy()

    # Transform logm1 (index 0) to m1
    paris_samples_transformed[:, 0] = 10**paris_samples[:, 0]
    dynesty_samples_transformed[:, 0] = 10**dynesty_samples[:, 0]

    # Transform logm2 (index 1) to m2
    paris_samples_transformed[:, 1] = 10**paris_samples[:, 1]
    dynesty_samples_transformed[:, 1] = 10**dynesty_samples[:, 1]

    # Transform cosqS (index 6) to qS
    paris_samples_transformed[:, 6] = np.arccos(paris_samples[:, 6])
    dynesty_samples_transformed[:, 6] = np.arccos(dynesty_samples[:, 6])

    # Parameter ranges from prior_transform function
    param_ranges = [
        [10**5.9998955889e+00, 10**6.0001044111e+00],
        [10**1.4770586003e+00, 10**1.4771839092e+00],
        [6.9991943608e-01, 7.0008056392e-01],
        [7.4995568044e+00, 7.5004431956e+00],
        [3.9997901137e-01, 4.0002098863e-01],
        [1.9727323083e+00, 2.0272676917e+00],
        [np.arccos(8.9037308426e-01), np.arccos(8.6479203952e-01)],
        [9.7750458220e-01, 1.0224954178e+00],
        [3.8232596200e-01, 4.1767403800e-01],
        [4.9151222865e-01, 5.0848777135e-01]
    ]

    # Parameter names for titles
    param_names = ['m1', 'm2', 'a', 'p0', 'e0', 'dist', 'qS', 'phiS','Phi_phi0', 'Phi_r0']

    # Visualization parameters
    bin_num = 50
    decay = 3  # For exponential smoothing

    def exponential_smoothing(hist, decay=1.0):
        """Apply exponential smoothing to histogram."""
        smoothed = np.zeros_like(hist)
        for i in range(len(hist)):
            weights_exp = np.exp(-decay * np.abs(np.arange(len(hist)) -i))
            weights_exp /= np.sum(weights_exp)  # normalize
            smoothed[i] = np.sum(hist * weights_exp)
        return smoothed

    # Set up the plot
    sns.set(style="white", context="talk")

    # Create subplot grid
    n_rows = (ndim + 1) // 2
    fig, axes = plt.subplots(n_rows, 2, figsize=(24, 3*n_rows))

    # Flatten axes for easier indexing
    if ndim == 1:
        axes = [axes]
    elif ndim <= 2:
        axes = axes.flatten()
    else:
        axes = axes.flatten()

    for i in range(ndim):
        ax = axes[i]

        # PARIS samples
        paris_param_samples = paris_samples_transformed[:, i]
        paris_hist, bin_edges = np.histogram(paris_param_samples,bins=bin_num, weights=paris_weights, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        paris_hist = exponential_smoothing(paris_hist, decay=decay)
        ax.plot(bin_centers, paris_hist, color='green', linewidth=2,label='PARIS')

        # Dynesty samples
        if i < dynesty_samples.shape[1]:
            dynesty_param_samples = dynesty_samples_transformed[:, i]
            dynesty_hist, bin_edges = np.histogram(dynesty_param_samples,bins=bin_num, weights=dynesty_weights, density=True)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            dynesty_hist = exponential_smoothing(dynesty_hist,decay=decay)
            ax.plot(bin_centers, dynesty_hist, color='blue', linewidth=2,label='Dynesty')

        # Add true value red line if provided
        if true_values is not None and i < len(true_values):
            ax.axvline(true_values[i], color='red', linestyle='--',linewidth=2,
                    label='True Value' if i == 0 else "")

        # Set x-axis limits to the prior ranges
        if i < len(param_ranges):
            ax.set_xlim(param_ranges[i][0], param_ranges[i][1])

        # Formatting with variable names
        param_name = param_names[i] if i < len(param_names) else f'Param {i+1}'
        ax.set_title(f'{param_name} Marginal Distribution', fontsize=14)
        ax.set_ylabel('Density', fontsize=12)
        ax.grid(True, alpha=0.3)

        # Add legend to first subplot
        if i == 0:
            ax.legend(fontsize=10, frameon=True)

    # Remove empty subplots if ndim is odd
    if ndim % 2 == 1 and ndim > 1:
        fig.delaxes(axes[-1])

    # Adjust layout
    plt.tight_layout()

    # Save the plot
    plot_filename = os.path.join(savepath,'compare_marginal.png')
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Marginal distribution comparison plot saved to: {plot_filename}")

    # Show plot if in interactive environment
    try:
        plt.show()
    except:
        pass

In [21]:
compare_marginal_linear(paris_sampler, '.', true_values=param_true_linear, dynesty_file='ns_likelihoodtest.pkl')

In [9]:

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI


# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 0.25     # Total time

print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": "cuda12x" # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": "cuda12x",  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs = {
    "force_backend": "cuda12x",  # Force GPU
    "pad_output": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs,
    use_gpu=use_gpu
)


Initializing waveform generator...
Creating GenerateEMRIWaveform class...


In [10]:
# Source parameters
m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 7.5
e0 = 0.4 
xI0 = 1.0
dist = 2 # Gpc
# Polar and azimuthal angles .. detector frame
# S = Solar system barycenter
# K = spin angular momentum of the MBH
qS = 0.5 
phiS = 1 
qK = 1 #fixed
phiK = phiS + np.pi/3
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # equatorial
Phi_r0 = 0.5


In [11]:
print('Generating data signal...')
data = waveform_gen(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0, T=T, dt=dt)
print('Done generating data signal.')


Generating data signal...
Done generating data signal.


In [12]:
# Change to the desired directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

# Add it to Python path
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
import GWfuncs
print('Setting up GWFuncs...')
gwf = GWfuncs.GravWaveAnalysis(T, dt)
print('Done setting up GWFuncs.')

Setting up GWFuncs...
Done setting up GWFuncs.


In [13]:
loglike(np.array([[np.log10(m1), np.log10(m2), a, p0, e0, dist, np.cos(qS), phiS, Phi_phi0, Phi_r0]]))

array([-1.39515088e-19])

In [5]:
# Change to the desired directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/sampling_test')

# Add it to Python path
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/sampling_test')

import pickle
with open('cov_matrix_new.pkl', 'rb') as f:
    cov_matrix = pickle.load(f)

In [6]:
sigmas = np.sqrt(np.diag(cov_matrix))
sigmas

array([1.51150496e-05, 9.07016210e-06, 2.68546395e-05, 1.47731858e-04,
       6.99620911e-06, 9.08923055e-03, 4.26350746e-03, 7.49847260e-03,
       5.89134600e-03, 2.82925712e-03])

In [7]:
sig = 1
a_var = np.linspace(a - sig*sigmas[2], a + sig*sigmas[2], 100)
p0_var = np.linspace(p0 - sig*sigmas[3], p0 + sig*sigmas[3], 100)
A, P0 = np.meshgrid(a_var, p0_var)
a_flat = A.flatten()
p0_flat = P0.flatten() 
params_batch = np.array([[np.log10(m1), np.log10(m2), a, p0, e0,dist, np.cos(qS), phiS, Phi_phi0, Phi_r0] for a, p0 in zip(a_flat, p0_flat)])

In [17]:
Z_flat = loglike(params_batch)
Z = Z_flat.reshape(A.shape)

KeyboardInterrupt: 

In [ ]:
plt.contourf(A, P0, Z)
plt.xlabel('a')
plt.ylabel('p0')
plt.title('Loglike')
plt.colorbar()
plt.savefig('loglike_contour_50std_100p.png')
plt.show()

In [ ]:
logm1_var = np.linspace(np.log10(m1) - sig*sigmas[0], np.log10(m1) + sig*sigmas[0], 10)
logm2_var = np.linspace(np.log10(m2) - sig*sigmas[1], np.log10(m2) + sig*sigmas[1], 10)
LM1, LM2 = np.meshgrid(logm1_var, logm2_var)
logm1_flat = LM1.flatten()
logm2_flat = LM2.flatten()
params_batch2 = np.array([[logm1, logm2, a, p0, e0,dist, np.cos(qS), phiS, Phi_phi0, Phi_r0] for logm1, logm2 in zip(logm1_flat, logm2_flat)])  
Z2_flat = loglike(params_batch2)
Z2 = Z2_flat.reshape(LM1.shape)
plt.contourf(LM1, LM2, Z2)
plt.xlabel('logm1')
plt.ylabel('logm2')
plt.title('Loglike')
plt.colorbar()
plt.savefig('loglike_contour_masses_50std_100p.png')
plt.show()

In [12]:
for i, name in enumerate(['logm1', 'logm2', 'a', 'p0', 'e0', 'dist', 'cosqS', 'phiS','Phi_phi0', 'Phi_r0']):
    param_value = param_true[i]
    result_low = param_value - sigmas[i]
    result_high = param_value + sigmas[i] 
    prior_low = param_value - 5*sigmas[i]
    prior_high = param_value + 5*sigmas[i]
    
    print(f"=== {name}: {param_value} +/- {sigmas[i]}")
    print(f"5-sigma prior range: [{prior_low:.10e}, {prior_high:.10e}]")

=== logm1: 6.0 +/- 1.5115049585719474e-05
5-sigma prior range: [5.9999244248e+00, 6.0000755752e+00]
=== logm2: 1.4771212547196624 +/- 9.070162101815586e-06
5-sigma prior range: [1.4770759039e+00, 1.4771666055e+00]
=== a: 0.7 +/- 2.6854639538955262e-05
5-sigma prior range: [6.9986572680e-01, 7.0013427320e-01]
=== p0: 7.5 +/- 0.00014773185757078157
5-sigma prior range: [7.4992613407e+00, 7.5007386593e+00]
=== e0: 0.4 +/- 6.9962091120260675e-06
5-sigma prior range: [3.9996501895e-01, 4.0003498105e-01]
=== dist: 2.0 +/- 0.009089230553048482
5-sigma prior range: [1.9545538472e+00, 2.0454461528e+00]
=== cosqS: 0.8775825618903728 +/- 0.004263507458024403
5-sigma prior range: [8.5626502460e-01, 8.9890009918e-01]
=== phiS: 1.0 +/- 0.00749847260096001
5-sigma prior range: [9.6250763700e-01, 1.0374923630e+00]
=== Phi_phi0: 0.4 +/- 0.005891345999342974
5-sigma prior range: [3.7054327000e-01, 4.2945673000e-01]
=== Phi_r0: 0.5 +/- 0.002829257116651039
5-sigma prior range: [4.8585371442e-01, 5.141462